# Capability 6: Semantic understanding: aliases, abbreviations, and typo correction

9/9 cases passed against a real, live LLM (gateway-configured model, see `.env`). Every code cell below is real, executable code -- the same `ask()` pattern as `notebooks/demo.ipynb` -- not a mockup; the attached output is what actually happened when this ran, captured via `scripts/run_live_capability_tests.py --capability 6`. Re-running this notebook (Restart Kernel & Run All) with a live key will make new real calls.

See `tests/live/cases/cap06_semantic_understanding.py` for these case definitions with their automated pass/fail checks, and `tests/live/live_capabilities_suite.py` for how they run as unittest assertions.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 01: zone abbreviation 'NA' for North America

**✅ PASS**

In [ ]:
_ = ask('NA rev Q1 2024?', '01')

01
USER: NA rev Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America revenue for Q1 2024 was **$3,593 million USD**.

## 02: zone alias 'EMEA' used directly (already canonical, sanity check)

**✅ PASS**

In [ ]:
_ = ask('EMEA revenue Q2 2025', '02')

02
USER: EMEA revenue Q2 2025

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA revenue for **Q2 2025** was **$2,489 million USD**.

Want me to pull the prior quarter or year-ago comparison for context?

## 03: country alias 'US' for United States, should roll up to North America

**✅ PASS**

In [ ]:
_ = ask("What's US revenue for Q1 2024?", '03')

03
USER: What's US revenue for Q1 2024?

[intent=data_query | sub_agents=['structured', 'unstructured']]
[citations: ['DOC-001', 'DOC-012', 'DOC-015', 'DOC-008', 'DOC-005']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **North America** zone (which includes United States) instead, plus any United States-specific commentary found in the documents.

AGENT: ## US Revenue — Q1 2024

**Short answer: AB InBev does not publicly disclose revenue at the country level, so a United States–only figure isn't available.** The structured database returned **$3,593M**, but that number is the **North America zone** total (which includes the US, Canada, and AB InBev's other North American operations) — not a US-only figure.

**What the evidence does show:**

| Metric (Q1 2024) | North America zone |
| --- | --- |
| Revenue | **$3,593M** |
| Organic revenue growth | −8.8% |
| Volume | 21,353K hL |
| Normalized EBITDA | $1,126M |


## 04: typo in a zone name ('Norht' for 'North')

**✅ PASS**

In [ ]:
_ = ask('What was the revenu for Norht America in Q1 2024?', '04')

04
USER: What was the revenu for Norht America in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: Revenue for **North America** in **Q1 2024** was **$3,593 million** (USD).

That's the only figure available from the retrieved KPI data for that zone/period.

Want me to check Q2 2024 for North America, or compare this against another zone for the same quarter?

## 05: typo in a KPI word ('EBTIDA' for 'EBITDA')

**✅ PASS**

In [ ]:
_ = ask("What was North America's EBTIDA margin in Q1 2025?", '05')

05
USER: What was North America's EBTIDA margin in Q1 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's EBITDA margin for Q1 2025 was **32.3%** (computed EBITDA margin).

## 06: KPI abbreviation 'rev' for revenue

**✅ PASS**

In [ ]:
_ = ask('EMEA rev 2025', '06')

06
USER: EMEA rev 2025

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA revenue for 2025 is **$9,502 million USD** (approx. **$9.5 billion**), based on the quarterly KPI data.

Note: this figure comes from quarterly-grain data for zone = EMEA, year = 2025. If you need a different breakdown (e.g., by month, brand, or country), let me know.

## 07: KPI shorthand 'margin' for EBITDA margin

**✅ PASS**

In [ ]:
_ = ask("What's the margin for North America in Q4 2025?", '07')

07
USER: What's the margin for North America in Q4 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: For North America in Q4 2025, the EBITDA margin was **28.0%**.

## 08: casual shorthand for a comparison ('NA vs EMEA')

**✅ PASS**

In [ ]:
_ = ask('NA vs EMEA revenue 2025', '08')

08
USER: NA vs EMEA revenue 2025

[intent=comparison | sub_agents=['structured']]

AGENT: ## North America vs EMEA revenue — 2025

Based on the quarterly KPI data for 2025, aggregated by zone:

| Zone | Revenue (USD million) |
| --- | --- |
| North America | $14,208 |
| EMEA | $9,502 |

**Key takeaway:** North America generated more revenue than EMEA in 2025 — **$14,208m vs $9,502m**.

**Scope note:** These figures come from the quarterly KPI grain for 2025, summed by zone, and cover only these two zones. They are not a company-wide total, and no other KPIs (e.g. volume, EBITDA) were included in this pull.

If you'd like, I can:
- Break these zones out **by quarter within 2025** to show seasonality.
- Compare against **2024** for the same zones to frame the year-over-year change.

## 09: brand alias/shorthand recognized (Bud for Budweiser)

**✅ PASS**

In [ ]:
_ = ask('How is Bud doing as a global brand?', '09')

09
USER: How is Bud doing as a global brand?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-012', 'DOC-014', 'DOC-004', 'DOC-015', 'DOC-011']]

AGENT: Based on the retrieved evidence, there isn’t a **Budweiser-specific** global performance figure to point to. Here’s what the data does and doesn’t support:

- Budweiser is named as one of AB InBev’s core “megabrands” (alongside Corona, Stella Artois, and Michelob Ultra) [DOC-012].
- As a group, those megabrands grew revenue **4.6% in FY2024** and **4.1% in FY2025**, both years **ahead of total company revenue growth** [DOC-012].
- However, the evidence does **not** break out Budweiser’s own revenue, volume, or market share at the global level. The structured database covers AB InBev results at zone/company level, not brand-level detail [DOC-015], and the other retrieved documents are zone- or peer-level commentary without brand-specific Bud figures.

So the honest answer is: **Budweiser appears to be part of a stron